# AI@UCI — Week 8: Linear Classification
### From Math to Code

**Goal:** make the mechanism from the slides executable.

> **feature space → score → prediction → mistake → update → repeat → test**

We will build a **binary linear classifier from scratch** using a perceptron-style update, then compare it with a library implementation.

This notebook mirrors the slide story:
1. A line becomes an equation.
2. The equation produces a score.
3. The score becomes a class prediction.
4. Wrong predictions create updates.
5. Repeated updates create training.
6. Unseen test data tells us whether the model generalizes.

## 0. Setup

We use:
- `numpy` — vectors and dot products
- `pandas` — a readable table
- `matplotlib` — feature-space and decision-boundary plots
- `scikit-learn` — train/test split, accuracy, and a reference `Perceptron`

The **from-scratch classifier** is the main activity. The sklearn version comes later so we can connect our implementation to a real ML API.

In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import Perceptron

RANDOM_STATE = 8
rng = np.random.default_rng(RANDOM_STATE)

## 1. Start where Week 3 left off: feature space

Each row is one sample with two numeric features:

- `feature_1`
- `feature_2`

The label is either `+1` or `-1`.

The data is synthetic so we can focus on the classifier instead of data cleaning.

In [ ]:
n_per_class = 45

positive = pd.DataFrame({
    "feature_1": rng.normal(2.2, 0.9, n_per_class),
    "feature_2": rng.normal(2.0, 0.9, n_per_class),
    "label": 1,
})

negative = pd.DataFrame({
    "feature_1": rng.normal(-2.0, 0.9, n_per_class),
    "feature_2": rng.normal(-1.8, 0.9, n_per_class),
    "label": -1,
})

df = pd.concat([positive, negative], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df.head()

### Completed example: visualize the feature space

This cell is already complete. Focus on the geometry, not the plotting syntax.

**Question:** where would you draw one straight line to separate the two classes?

In [ ]:
for label, group in df.groupby("label"):
    plt.scatter(
        group["feature_1"],
        group["feature_2"],
        label=f"Class {label:+d}",
        alpha=0.8
    )

plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Feature Space")
plt.legend()
plt.show()

## 2. Separate features from labels

As in Week 3:
- `X` = model inputs / features
- `y` = correct labels

In [ ]:
X = df[["feature_1", "feature_2"]].to_numpy()
y = df["label"].to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)

## 3. Split before learning

The model may learn from the **training set**.

The **test set stays unseen** until evaluation.

In [ ]:
# TODO 1:
# Create an 80/20 train-test split.
# Use RANDOM_STATE and stratify=y.

X_train, X_test, y_train, y_test = ...

print("training examples:", len(X_train))
print("test examples:", len(X_test))

## 4. A line becomes a score

The slide equation was:

$w_1x_1 + w_2x_2 + b = 0$

For a point $x$, we compute:

$z = w^T x + b$

The sign of `z` tells us which side of the boundary the point is on.

### Completed example: compute one score

This is the exact pattern from the slides.

In [ ]:
w_demo = np.array([2.0, -1.0])
b_demo = -1.0
x_demo = np.array([3.0, 2.0])

z_demo = np.dot(w_demo, x_demo) + b_demo

print("score:", z_demo)
print("prediction:", 1 if z_demo > 0 else -1)

### Your turn: do the same thing for a new point

In [ ]:
w = np.array([1.5, 0.5])
b = -0.25
x = np.array([-1.0, 2.0])

# TODO 2:
# 1. Compute z = w^T x + b
# 2. Predict +1 if z > 0, otherwise -1

z = ...
prediction = ...

print("score:", z)
print("prediction:", prediction)

## 5. Turn the score into a reusable prediction function

In [ ]:
def predict_one(x, w, b):
    # TODO 3:
    # Compute the score and return +1 or -1.
    z = ...
    return ...

def predict(X, w, b):
    return np.array([predict_one(x, w, b) for x in X])

## 6. One mistake → one update

For a misclassified example:

$w \leftarrow w + \eta yx$

$b \leftarrow b + \eta y$

The example and its true label push the boundary in a better direction.

### Completed example: one manual update

Study the pattern first.

In [ ]:
w_example = np.array([0.2, -0.1], dtype=float)
b_example = 0.0
x_example = np.array([2.0, 1.0])
y_true_example = 1
learning_rate = 0.1

y_pred_example = predict_one(x_example, w_example, b_example)

if y_pred_example != y_true_example:
    w_example = w_example + learning_rate * y_true_example * x_example
    b_example = b_example + learning_rate * y_true_example

print("updated w:", w_example)
print("updated b:", b_example)

### Your turn: repeat the update pattern

This follows the workshop guideline: first study a completed example, then implement a similar one yourself.

In [ ]:
w_try = np.array([-0.2, 0.1], dtype=float)
b_try = 0.0
x_try = np.array([2.5, 2.0])
y_true_try = 1
learning_rate = 0.1

y_pred_try = predict_one(x_try, w_try, b_try)

# TODO 4:
# If the prediction is wrong, update w_try and b_try.
if y_pred_try != y_true_try:
    w_try = ...
    b_try = ...

print("prediction before update:", y_pred_try)
print("updated w:", w_try)
print("updated b:", b_try)

## 7. Learning is a loop

Now we combine the same pieces:

> **predict → check → update → repeat**

One full pass through the training set is an **epoch**.

In [ ]:
def train_perceptron(X_train, y_train, learning_rate=0.05, epochs=15):
    # Start with a deliberately imperfect boundary.
    w = np.array([-0.5, 0.8], dtype=float)
    b = 0.0

    mistakes_per_epoch = []
    history = [(w.copy(), b)]

    for epoch in range(epochs):
        mistakes = 0

        for x, y_true in zip(X_train, y_train):
            y_pred = predict_one(x, w, b)

            # TODO 5:
            # If the model is wrong:
            # 1. update w
            # 2. update b
            # 3. count the mistake
            if y_pred != y_true:
                w = ...
                b = ...
                mistakes += ...

        mistakes_per_epoch.append(mistakes)
        history.append((w.copy(), b))

    return w, b, mistakes_per_epoch, history

w, b, mistakes, history = train_perceptron(X_train, y_train)

print("learned w:", w)
print("learned b:", b)
print("mistakes per epoch:", mistakes)

### Completed visualization: mistakes during training

Do **not** expect every real training curve to decrease perfectly every epoch.
This plot simply lets us inspect what happened in this run.

In [ ]:
plt.plot(range(1, len(mistakes) + 1), mistakes, marker="o")
plt.xlabel("epoch")
plt.ylabel("mistakes")
plt.title("Mistakes During Training")
plt.show()

## 8. Watch the boundary move

The boundary is:

$w_1x_1 + w_2x_2 + b = 0$

Solving for $x_2$:

$x_2 = -\frac{w_1x_1 + b}{w_2}$

The helper below is completed so we can focus on the learned model.

In [ ]:
def plot_boundary(X, y, w, b, title):
    plt.figure(figsize=(6, 5))

    for label in [-1, 1]:
        group = X[y == label]
        plt.scatter(
            group[:, 0],
            group[:, 1],
            label=f"Class {label:+d}",
            alpha=0.8
        )

    xs = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)

    if abs(w[1]) < 1e-12:
        plt.axvline(-b / w[0], label="decision boundary")
    else:
        ys = -(w[0] * xs + b) / w[1]
        plt.plot(xs, ys, label="decision boundary")

    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.title(title)
    plt.legend()
    plt.show()

plot_boundary(X_train, y_train, history[0][0], history[0][1], "Before Training")
plot_boundary(X_train, y_train, w, b, "After Training")

## 9. Does it work on new data?

Training accuracy tells us how well the model fits examples it saw.

Test accuracy asks the more important question:

> **Does the learned boundary generalize to unseen examples?**

In [ ]:
# TODO 6:
# Predict on both sets and compute accuracy.

train_pred = ...
test_pred = ...

train_accuracy = ...
test_accuracy = ...

print("train accuracy:", round(train_accuracy, 3))
print("test accuracy:", round(test_accuracy, 3))

## 10. Connect our implementation to a real ML library

We already built the mechanism ourselves.

Now look at how compact the same workflow becomes with `sklearn`.

In [ ]:
sk_model = Perceptron(
    max_iter=1000,
    eta0=0.05,
    random_state=RANDOM_STATE
)

sk_model.fit(X_train, y_train)

sk_test_pred = sk_model.predict(X_test)

print("sklearn test accuracy:", round(accuracy_score(y_test, sk_test_pred), 3))
print("sklearn weights:", sk_model.coef_)
print("sklearn bias:", sk_model.intercept_)

## 11. Final challenge: when one line is not enough

A linear classifier can only create a **linear decision boundary**.

Try changing the dataset or drawing a pattern that cannot be separated by one straight line.

**Question:** what kind of model might handle multiple regions better?

That is the bridge to **Decision Trees & Ensembling**.

## Reflection

Before leaving, make sure you can explain these in plain English:

1. What does $z = w^T x + b$ represent?
2. How does the sign of `z` become a prediction?
3. What do `w` and `b` control geometrically?
4. Why do we update only after a mistake in this perceptron-style algorithm?
5. What is an epoch?
6. Why do we still need a test set after training accuracy is high?
7. What is the main limitation of a linear classifier?